# M02 — Gravity prisms and the inverse acceptance boundary

**Level 3 · Prerequisites: foundations notebook; read time: 35 minutes.**

## Objectives

Run the product SimPEG/Geoana prism forward operator on its frozen synthetic test control and compare with the independent Choclo oracle. The current retained-48 native direction gate failed; dependent inverse checks are blocked. This notebook records that negative result.

## Setup

Use the already reviewed Python 3.12 pipeline environment. This notebook installs nothing.

Import repository code and confirm its source file.

In [ ]:
import runpy,sys
from pathlib import Path
import numpy as np
root=Path.cwd().parents[2];sys.path.insert(0,str(root/'data-pipeline'))
import gravity_forward
# Load the product's existing synthetic control, not a field survey.
fixture=runpy.run_path(str(root/'tests/numerics/test_gravity_forward.py'))
print(gravity_forward.ENGINE)

## Data

Run the product SimPEG/Geoana prism forward operator on its frozen synthetic test control and compare with the independent Choclo oracle. The current retained-48 native direction gate failed; dependent inverse checks are blocked. This notebook records that negative result.

**Files:** tests/numerics/test_gravity_forward.py, docs/design/features/m02-prism-operator/validation.md

## Method

For an exterior receiver, upward acceleration is g_z=G∫_V Δρ(z′−z)/|r−r′|³ dV. The discrete operator d=Jρ uses density contrast kg m⁻³, prediction mGal and J in mGal/(kg m⁻³). Positive buried mass gives negative upward gz. The inverse adds declared errors, reference, bounds and regularization; potential-field models are non-unique.

Execute the product method on the declared control.

In [ ]:
request=fixture['request']()
# Compare production prediction with a separate physical prism implementation.
result=gravity_forward.forward_gravity(request)
oracle=fixture['choclo_oracle'](request)
print({'product_up_mGal':result['gz_up_mgal'].tolist(),'oracle_up_mGal':oracle.tolist(),'max_abs_difference_mGal':float(np.max(np.abs(result['gz_up_mgal']-oracle)))})

Adversarial or negative control: isolate one assumption or violate one declared physical constraint.

In [ ]:
from copy import deepcopy
invalid=deepcopy(request);invalid['receivers_m']=np.array([[0.0,0.0,-50.0]])
try:
    # Closed-mesh interior receivers violate the forward contract.
    gravity_forward.forward_gravity(invalid)
except ValueError as error:
    print('Expected geometry refusal:',str(error))
else:
    raise AssertionError('Interior receiver was accepted')

## State of the art

SimPEG/Geoana is the production forward engine ([Cockett et al., 2015](https://doi.org/10.1016/j.cageo.2015.09.015)); the product validation uses an independent analytic prism oracle. Forward agreement is not density recovery. The inverse remains unaccepted because the retained-48 direction gate failed; the positive-null and first-fold checks did not run.

## Results

Only bounded synthetic forward behavior is evaluated. The invalid receiver must be refused. No inversion, field fit, held-out result or host capacity is claimed; preserve the failed source-bound gate before any inverse claim.

## Summary

A validated forward kernel can coexist with an unresolved inverse. Do not turn a retained solver failure into success with plots or changed tolerances.




## References

- https://doi.org/10.1016/j.cageo.2015.09.015